In [ ]:
import numpy as np
import pandas as pd
import os
import requests
import json
import dotenv


: 

In [ ]:
dotenv.load_dotenv()

In [ ]:
congresskey = os.getenv('congresskey')

There are five pieces of information you need to navigate any API and its poor documentation
1. How you get the API key (or another credential) to get access to the data (check)
2. Root: the beginning of the URL for any endpoint contained within the API 
A URL: `root / endpoint ? parameters` like youtube share ID
3. Endpoint: the middle of the URL, specifies which data store on the API to access
4. Parameters: ways to control what exactly is returned when you access an endpoint. Two kinds of parameters:
* Query parameters (appear after the ?, but reqests formats this for you using the `params` argument)
* Path parameters -- supposed to be entered as part of the endpoint itself
5. How to supply your API key back to the API to prove that you are you

In [ ]:
root = 'https://api.congress.gov/v3'  # no slash at end of root bc slash is typically part of endpoint

endpoint = '/member'

params = {'format': 'json', 'currentMember': 'true', 'api_key': congresskey}

In [ ]:
r = requests.get(root + endpoint, params=params)
r

In [ ]:
r.text

In [ ]:
r.json()

1. User-Agent strings/headers

A piece of text you give to the data server to tell them a bit about who you are and what your system is

2. Navigate a JSON

3. Pagination and rate limits

In [ ]:
botname = 'ds6600'
version = '0.0'
purpose = 'Class demonstration'
email = 'gsu4aq@virginia.edu'
useragent = f'{botname}/{version} ({purpose}; {email}) python-requests/{requests.__version__}'
headers = {'User-Agent': useragent}
headers  # useragent

In [ ]:
root = 'https://api.congress.gov/v3'
endpoint = '/member'
params = {'format': 'json',
          'currentMember': True,
          'api_key': congresskey}

r = requests.get(root + endpoint, params=params, headers=headers)  # the equals makes sure everything goes to the right place
r

In [ ]:
myjson = r.json()
myjson

In [ ]:
myjson.keys()

In [ ]:
myjson['members']

In [ ]:
len(myjson['members'])

In [ ]:
myjson['members'][0]['terms']['item'][0]['chamber']

Record path -- somewhere in the JSON is a list where each item is one row of the dataframe you want to build. The record path is the sequence of keys to call to arrive at this list.

In this case the record path is: just members

In [ ]:
congress_df = pd.json_normalize(myjson, record_path=['members'])
congress_df

In [ ]:
params['limit'] = '250'
# params DO NOT WRITE THIS IT WILL EXPOSE YOUR KEY !!

In [ ]:
r = requests.get(root + endpoint, params=params, headers=headers)
myjson = r.json()
congress_df =  pd.json_normalize(myjson, record_path=['members'])
# pd.set_option(max rows) DO NOT WRITE THIS IT WILL CRASH YOUR COMPUTER IF U HAVE A LARGE AMNT OF ROWS !!
congress_df

In [ ]:
params['offset'] = '500'
r = requests.get(root + endpoint, params=params, headers=headers)
myjson = r.json()
myjson
# congress_df =  pd.json_normalize(myjson, record_path=['members'])
# congress_df

In [ ]:
i = 0
# still_members = True
congress_json = []
while i < total_count:  # > 0
    params['offset'] = i
    r = requests.get(root + endpoint, params=params, headers=headers)
    myjson = r.json()
    # still_members = len(myjson['members']) > 0
    total_count = myjson['pagination']['count']
    congress_json.extend(myjson['members'])
    i = i + 250

In [ ]:
len(congress_json)

In [ ]:
congress_df = pd.json_normalize(congress_json)
congress_df

In [ ]:
mcguire_df = congress_df.query("state == 'Virginia' & district == 5")

In [ ]:
bioguideId = mcguire_df.reset_index()['bioguideId'][0]  # why?

In [ ]:
endpoint = f'/member/{bioguideId}'  # member not memberS !!!!!
endpoint

In [ ]:
params = {'api_key': congresskey, 'format': 'json'}
r = requests.get(root + endpoint, params=params, headers=headers)
r

In [ ]:
r.json()

In [ ]:
bioguideId

In [ ]:
endpoint = f'/member/{bioguideId}/sponsored-legislation'

params = {'api_key': congresskey, 'format': 'json'}
r = requests.get(root + endpoint, params=params, headers=headers)
r

In [ ]:
mcguire_bills_df = pd.json_normalize(r.json(), record_path=['sponsoredLegislation'])
pd.set_option('display.max_rows', 20)
mcguire_bills_df['title']

In [ ]:
mcguire_bills_df.loc[11]  # why not iloc?